# 농수산식품

In [0]:
from pyspark.sql.functions import concat_ws, col, lit

df_agrofood = spark.table("bronze_api.agrofood_perday.perday")

agrofood_standard = df_agrofood.select(
    col("exmn_ymd").alias("날짜"),
    lit("agrofood").alias("출처"),
    col("item_nm").alias("재료명"),
    col("vrty_nm").alias("세부속성"),
    concat_ws("", col("unit_sz"), col("unit")).alias("단위"),
    col("unit_sz").alias("단위_수치"),
    col("unit").alias("단위_문자"),
    col("ctgry_nm").alias("카테고리"),
    col("mrkt_nm").alias("업태"),
    col("sgg_nm").alias("지역"),
    col("exmn_dd_prc").alias("가격"),
    col("grd_nm").alias("등급"),
    col("collect_time").alias("수집시간")
    
)

display(agrofood_standard)


In [0]:
# agrofood_standard.write.mode("overwrite").option("header", "true").csv("/Volumes/silver/agrofood/agrofood_normalized")

In [0]:
from pyspark.sql.functions import when

agrofood_standard = agrofood_standard.withColumn(
    "재료명",
    when(
        col("재료명").like("%호박%"),
        col("세부속성")
    ).otherwise(col("재료명"))
)
display(agrofood_standard.filter(col("재료명").like("%호박%")))
# 애호박, 단호박, 쥬키니

In [0]:
from pyspark.sql.functions import when

agrofood_standard = agrofood_standard.withColumn(
    "재료명",
    when(
        col("세부속성").isin("거봉", "샤인머스켓"),
        col("세부속성")
    ).otherwise(col("재료명"))
)
display(agrofood_standard.filter(col("재료명").like("%포도%") | col("세부속성").like("%포도%")))

In [0]:
agrofood_standard = agrofood_standard.withColumn(
    "재료명",
    when(
        col("재료명").like("파"),
        col("세부속성")
    ).otherwise(col("재료명"))
)
display(agrofood_standard.filter(col("재료명").like("%파")))
# 대파, 쪽파

In [0]:
display(agrofood_standard)
agrofood_standard.write.mode("overwrite").saveAsTable("silver.agrofood.`agrofood_normalized2`")

In [0]:
display(agrofood_standard)